# TranslationTurbo: Headless GPU Worker
This notebook connects to your Master server and processes video translation tasks using Google's free T4 GPU.

In [ ]:
# @title 1. Environment Setup
%pip install celery redis requests moviepy yt-dlp openai faster-whisper loguru sqlalchemy pydantic-settings
!apt-get update && apt-get install -y ffmpeg
import os
import sys
print("Environment ready.")

In [ ]:
# @title 2. Worker Configuration
MASTER_IP = "YOUR_ORACLE_IP" # @param {type:"string"}
WORKER_NAME = "colab-gpu-worker-1"

os.environ['REDIS_HOST'] = MASTER_IP
print(f"Connecting to Master at: {MASTER_IP}")

In [ ]:
# @title 3. Start Headless Worker
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_FORK.git" # @param {type:"string"}
import shutil
%cd /content
if os.path.exists('/content/mpt'): shutil.rmtree('/content/mpt')
!git clone {REPO_URL} /content/mpt
if os.path.exists('/content/mpt/backend'):
    %cd /content/mpt/backend
    !pip install -e .
    # Set PYTHONPATH and ensure REDIS_HOST is set
    %env PYTHONPATH=/content/mpt/backend
    %env REDIS_HOST={MASTER_IP}
    # Test connection before starting
    !python3 -c "import redis; r=redis.Redis(host='{MASTER_IP}', port=6379); print('Redis Ping:', r.ping())"
    !python3 -m celery -A app.celery_app:celery_app worker --loglevel=info -n {WORKER_NAME} -Q default,gpu_tasks
else:
    print('Error: /content/mpt/backend not found. Check your REPO_URL structure.')